# Here the experimets conducted wiht Gemini 2.5 flash is conducted

In [1]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
from data_processing.data_analysis import select_problem_sample_for_model 
gpt5_nano="gemini-2.5-flash (think)"
df = select_problem_sample_for_model(gpt5_nano)
df

,Unnamed: 0,source,problem,competition,unique_problem_label,correct,parsed_answer,gold_answer,output_cost_per_tokens,problem_idx,cost,output_tokens,input_tokens,answer,user_message,idx_answer,model_config,model_name,input_cost_per_tokens,ten_percentile_group
3736,1828,NaN,Suppose $\triangle ABC$ has angles $\angle BAC...,MathArena/aime_2025_outputs,MathArena/aime_2025: 20,False,300,336,3.5,20,0.101926,29094.0,644.0,"Let $A=84^\circ$, $B=60^\circ$, $C=36^\circ$ b...","Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,4
3752,1844,NaN,"Let $ABCDE$ be a convex pentagon with $AB=14$,...",MathArena/aime_2025_outputs,MathArena/aime_2025: 14,False,47,60,3.5,14,0.112694,32191.0,169.0,"Let the vertices be $A, B, C, D, E$. The side ...","Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,1
3780,1872,NaN,Alex divides a disk into four quadrants with t...,MathArena/aime_2025_outputs,MathArena/aime_2025: 13,False,79,204,3.5,13,0.093253,26639.0,109.0,"Let $D$ be the disk. Initially, the disk is di...","Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,2
13604,1856,NaN,"Albert writes $2025$ numbers $a_{1}, \ldots, a...",MathArena/hmmt_feb_2025_outputs,MathArena/hmmt_feb_2025: 18,False,201,\frac{2025}{101},3.5,18,0.152056,43437.0,178.0,Let $N=2025$ and $T=100$. The numbers are $a_1...,"Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,3
18644,776,NaN,Consider a $4 \times 4$ grid of squares. We pl...,MathArena/cmimc_2025_outputs,MathArena/cmimc_2025: 16,False,40,256,3.5,16,0.115904,33112.0,80.0,Let the grid squares be denoted by $c_{ij}$ fo...,"Please reason step by step, and put your final...",0,gemini/gemini-flash-2.5,gemini-2.5-flash (think),0.15,5


## statistics dataframe

In [2]:
import pandas as pd


UNIQUE_PROBLEM_LABEL = "unique_problem_label"
DIFFICULTY = "ten_percentile_group"
METHOD = "method"
TOTAL_TOKENS = "total_tokens"
CORRECT = "correct"
PARAMS = "params"
RESULT_DESC = "result_description"
cols = [UNIQUE_PROBLEM_LABEL, DIFFICULTY, METHOD, TOTAL_TOKENS, CORRECT, PARAMS, RESULT_DESC]

df_stats = pd.DataFrame(columns=cols)

# Problem 1, difficulty level 5

In [3]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==5].iloc[0][[UNIQUE_PROBLEM_LABEL, "answer", "gold_answer",  DIFFICULTY, "problem"]]
df_pruned


unique_problem_label                             MathArena/cmimc_2025: 16
answer                  Let the grid squares be denoted by $c_{ij}$ fo...
gold_answer                                                           256
ten_percentile_group                                                    5
problem                 Consider a $4 \times 4$ grid of squares. We pl...
Name: 18644, dtype: object

In [4]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

REFLEXION = "reflexion"
TOT = "tot"
RS_BASIC = "rejection_sampling_basic"
RS_SUM = "rejection_sampling_summarized"

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5},
 'tot': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5}}

In [5]:
import textwrap

first_problem_description = df_pruned["problem"]

first_problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=first_problem_description, width=80))
print("answer", first_problem_gold_answer)



Consider a $4 \times 4$ grid of squares. We place coins in some of the grid
squares so that no two coins are orthogonally adjacent, and each $2 \times 2$
square in the grid has at least one coin. How many ways are there to place the
coins?
answer 256


## reflexiton [No]

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=first_problem_description,
    answer=first_problem_gold_answer
)

from models.google_api import GoogleClient, google_user_format
model_name="gemini-2.5-flash"


client = GoogleClient()

model = client.select_model(
  model_name=model_name

)

eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3,
    user_format=google_user_format
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
The problem asks us to count the number of ways to place coins in a $4 \times 4$ grid such that two conditions are met:
1.  No two coins are orthogonally adjacent.
2.  Each $2 \times 2$ square in the grid has at least one coin.

Let's represent the grid cells by $x_{i,j}$ where $1$ means a coin and $0$ means no coin.

**Condition 1: No two coins are orthogonally adjacent.**
This condition applies both horizontally and vertically.
For any row $(x_{i,1}, x_{i,2}, x_{i,3}, x_{i,4})$, no two adjacent cells can have coins. This is an independent set problem on a path graph $P_4$. The number of ways to choose coins for a row is $F_{4+2} = F_6 = 8$. Let these 8 patterns be $P_0$ to $P_7$:
$P_0 = 0000$ (0 coins)
$P_1 = 1000$ (1 coin)
$P_2 = 0100$ (1 coin)
$P_3 = 0010$ (1 coin)
$P_4 = 0001$ (1 coin)
$P_5 = 1010$ (2 coins)
$P_6 = 0101$ (2 coins)
$P_7 = 1001$ (2 coins)

Let $r_k$ denote the pattern of row $k$. If $r_i$ and $r_{i+1}$ are two adjacent rows, then they cannot have 

In [6]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 49787
problem_rows[REFLEXION][CORRECT] = False
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"


Not correct

## TOT [No]

In [8]:


from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from google import genai
model_name="gemini-2.5-flash"
config = ToTConfig(
    key_env_name="GEMINI_API_KEY",
    endpoint_env_name= None,
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=None,
    client_type=genai.Client
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=first_problem_description, answer=first_problem_gold_answer)

print(ys)


>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
vote no match: ['The problem requires finding the number of ways to place coins in a $4 \\times 4$ grid such that:\n1.  No two coins are orthogonally adjacent.\n2.  Each $2 \\times 2$ square in the grid has at least one coin.\n\nLet\'s analyze the choices:\n\n**Choice 1 Analysis:**\n*   **Step 1: Checkerboard Coloring and Condition 1:** The analysis correctly identifies that Condition 1 implies that if a coin is on a square, its four neighbors (of the opposite color) must be empty. This means that if coins are placed on squares of one color (e.g., all black), Condition 1 is automatically satisfied, as no two squares of the same color are orthogonally adjacent.\n*   **Step 2: Monochromatic $2 \\times 2$ squares:** The analysis correctly deduces that any $2 \\times 2$ square must contain coins of only one color. If it contained a black coin and

In [9]:
from baselines.tot.tree_of_thought_llm_master.src.tot.models import google_usage
usage = google_usage()
print(usage)

{'total_token_count': 1612774}


In [7]:
problem_rows[TOT][TOTAL_TOKENS] = 1612774
problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=4"

Gemini 2.5 flash did not get the right answer

## our method: Basic form [No]

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation_google, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

first_problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

messages, raw, path = conversation_google(model=model, name="gemini_25_flash_think_Problem1_attmp1" ,n_steps=20, problem=first_problem)
print(model.compute_token_cost())


STEP 0: 


Agent: Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize an

In [8]:
usage = {
    "total_token_count": [
        34528, 3706, 37396, 8238, 39730, 13055, 36460, 17144, 28345, 19890,
        37640, 20850, 39593, 22812, 38188, 24459, 40350, 28282, 39401, 30056,
        37325,
    ],

    "candidates_token_count": [
        3122, 93, 4462, 88, 4737, 93, 3993, 93, 2653, 99,
        689, 131, 1808, 124, 1659, 134, 3433, 143, 1860, 146,
        1604,
    ],

    "prompt_token_count": [
        489, 3471, 3706, 8028, 8258, 12855, 13090, 16943, 17178, 19691,
        19932, 20481, 20754, 22422, 22688, 24207, 24483, 27776, 28061, 29781,
        29877,
    ],

    "thoughts_token_count": [
        30917, 142, 29228, 122, 26735, 107, 19377, 108, 8514, 100,
        17019, 238, 17031, 266, 13841, 118, 12434, 363, 9480, 129,
        5844,
    ],

    "cached_content_token_count": [
        None, None, None, None, None, None, None, 12269, 12270, 16364,
        16365, 19435, 19436, 20461, 20462, 21488, 22512, 23538, 23539, 27635,
        None,
    ],
}
sums = {k: sum(v for v in vals if v is not None) for k, vals in usage.items()}
print(sums)

{'total_token_count': 597448, 'candidates_token_count': 31164, 'prompt_token_count': 374171, 'thoughts_token_count': 192113, 'cached_content_token_count': 255774}


In [9]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 597448
problem_rows[RS_BASIC][CORRECT] = False
problem_rows[RS_BASIC][PARAMS] = "n_steps=20"


In [14]:
problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5,
  'total_tokens': 49787,
  'correct': False,
  'params': 'max_attempt=3'},
 'tot': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5,
  'total_tokens': 597448,
  'correct': False,
  'params': 'n_steps=20'},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5}}

## Our method: Summarized form [2nd]

In [ ]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_google
from multi_agent.multi_agent import Role, Problem, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient
from importlib import reload
reload(summerized_conv)

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=first_problem_description, 
  answer=first_problem_gold_answer
  
)

msg = summarized_rejection_sampling_google(
  model=model,
  name=f"{model_name}_problem_1",
  n_steps=10,
  problem=problem
)

rank_google_answer(model=model, conversation=msg)

Iteration 0

Solver: 
[
(104, Calculated ways with only E-cells (52) and only O-cells (52), assuming no mixed configurations exist),
(160, Sum of pure E (52), pure O (52), and mixed derived from central two-diagonal coins (56), but this was an intermediate thought, not a complete mixed calculation),
]

Let the grid squares be denoted by $(i,j)$ for $1 \le i,j \le 4$. A square contains a coin if $x_{ij}=1$, otherwise $x_{ij}=0$.

The conditions are:
1.  **No two coins are orthogonally adjacent:** If $x_{ij}=1$, then $x_{i \pm 1, j}=0$ and $x_{i, j \pm 1}=0$ (within grid boundaries). This implies that coins must be placed in cells that form an independent set in the grid graph.
2.  **Each $2 \times 2$ square in the grid has at least one coin:** For any $1 \le i,j \le 3$, $x_{ij} + x_{i,j+1} + x_{i+1,j} + x_{i+1,j+1} \ge 1$.

Condition 1 is crucial. If $x_{ij}=1$, it implies its four orthogonal neighbors must be empty. This effectively creates "blocked" cells around each coin. Due to this

('Okay, Solver. I will now go through each of your submitted answers, arguing how they could be true given the problem constraints and your calculation history, and then rank them based on their plausibility.\n\n---\n\n**Understanding the Core Components (My Current Best Estimates):**\n\n1.  **Pure Configurations (all coins on same-colored cells):** My calculations have consistently yielded **104** ways (52 for white cells, 52 for black cells). This component has been extensively verified and is considered highly robust. Any deviation from 104 for the pure count requires a compelling argument against this established base.\n2.  **Mixed Configurations (coins on both white and black cells):** This is where the majority of complexity and error has resided. My strategy typically involved analyzing the central $2 \\times 2$ square:\n    *   **Exactly one coin in the central $2 \\times 2$:** This has been the most volatile category, with counts ranging from 96 to 208.\n    *   **Exactly two 

Summerarized form ranked te correct answer 2. 

In [10]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  538780
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][PARAMS] =  "n_steps=10"
problem_rows[RS_SUM][RESULT_DESC] =  "2nd"


In [11]:
problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5,
  'total_tokens': 49787,
  'correct': False,
  'params': 'max_attempt=3'},
 'tot': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5,
  'total_tokens': 1612774,
  'correct': False,
  'params': 'n_evaluate_sample=4,\nn_select_sample=4,\nn_generate_sample=4,\nsteps=4,'},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5,
  'total_tokens': 597448,
  'correct': False,
  'params': 'n_steps=20'},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/cmimc_2025: 16',
  'ten_percentile_group': 5,
  'total_tokens': 538780,
  'correct': False,
  'params': 'n_steps=10',
  'result_description': '2nd'}}

## Token cost

In [11]:

for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats


,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/cmimc_2025: 16,5,reflexion,49787,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 16,5,tot,1612774,False,all=4,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_basic,597448,False,n_steps=20,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_summarized,538780,False,n_steps=10,2nd


# Problem 2, difficulty level 4

In [12]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==4].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)


Suppose $\triangle ABC$ has angles $\angle BAC = 84^\circ$, $\angle ABC =
60^\circ$, and $\angle ACB = 36^\circ$. Let $D$, $E$, and $F$ be the midpoints
of sides $\overline{BC}$, $\overline{AC}$, and $\overline{AB}$, respectively.
The circumcircle of $\triangle DEF$ intersects $\overline{BD}$, $\overline{AE}$,
and $\overline{AF}$ at points $G$, $H$, and $J$, respectively. The points $G$,
$D$, $E$, $H$, $J$, and $F$ divide the circumcircle of $\triangle DEF$ into six
minor arcs, as shown. Find $\wideparen{DE} + 2 \cdot \wideparen{HJ} + 3 \cdot
\wideparen{FG}$, where the arcs are measured in degrees.
\begin{tikzpicture}[scale=1.2]      \coordinate (B) at (0,0);      \coordinate
(C) at (6,0);      \coordinate (A) at (1.78,3.07);            \coordinate (D) at
($(B)!0.5!(C)$);      \coordinate (E) at ($(A)!0.5!(C)$);      \coordinate (F)
at ($(A)!0.5!(B)$);            \draw (A) -- (B) -- (C) -- cycle;        \fill
(A) circle (2pt) node[above] {$A$};      \fill (B) circle (2pt) node[below]
{

In [13]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4},
 'tot': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4}}

### Reflexion [Yes]

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
problem =  Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

from models.google_api import GoogleClient, google_user_format
model_name="gemini-2.5-flash"


client = GoogleClient()

model = client.select_model(
  model_name=model_name

)

eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3,
    user_format=google_user_format
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
The problem asks for the sum $\wideparen{DE} + 2 \cdot \wideparen{HJ} + 3 \cdot \wideparen{FG}$, where the arcs are measured in degrees on the circumcircle of $\triangle DEF$.

First, let's identify the key geometric figures and points:
1.  $\triangle ABC$ has angles $A=84^\circ$, $B=60^\circ$, $C=36^\circ$. Since all angles are acute, $\triangle ABC$ is an acute triangle.
2.  $D, E, F$ are the midpoints of $BC, AC, AB$ respectively. $\triangle DEF$ is the medial triangle of $\triangle ABC$.
3.  The circumcircle of $\triangle DEF$ is the nine-point circle of $\triangle ABC$. Let's call this circle $\Omega$.
4.  $G, H, J$ are points on $\Omega$.
    *   $G$ is on $\overline{BD}$ (part of side $BC$). Since $D$ is the midpoint of $BC$ and $D \in \Omega$, $G$ must be the other intersection of the line $BC$ with $\Omega$. For an acute triangle, the feet of the altitudes lie on the nine-point circle. $G$ is therefore the foot of the altitude from $A$ to $BC$. Let $H_A$ be 

Refleciton got the rigt answer, 336

In [14]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 17591 + 7803 + 14288 + 9462 + 17821

problem_rows[REFLEXION][CORRECT] = True
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"


In [21]:
problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4,
  'total_tokens': 66965,
  'correct': True,
  'params': 'max_attempt=3'},
 'tot': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/aime_2025: 20',
  'ten_percentile_group': 4}}

### TOT [Yes]

In [4]:


from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from google import genai
model_name="gemini-2.5-flash"
config = ToTConfig(
    key_env_name="GEMINI_API_KEY",
    endpoint_env_name= None,
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=None,
    client_type=genai.Client
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import google_usage
usage = google_usage()
print(usage)

path c:\Users\Thomas\Documents\Datalogi\Kandidat\9. semester\9. semester notes\NLP\Project\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
vote no match: ["The problem requires us to calculate $\\wideparen{DE} + 2 \\cdot \\wideparen{HJ} + 3 \\cdot \\wideparen{FG}$ on the circumcircle of $\\triangle DEF$.\n\n**1. Identify the points and the circle:**\n*   $\\triangle ABC$ has angles $A=84^\\circ$, $B=60^\\circ$, $C=36^\\circ$. It's an acute triangle.\n*   $D, E, F$ are the midpoints of $BC, AC, AB$ respectively. $\\triangle DEF$ is the medial triangle.\n*   The circumcircle of $\\triangle DEF$ is the nine-point circle of $\\triangle ABC$. This circle passes through the midpoints of the sides ($D, E, F$) and the feet of the altitudes ($H_A, H_B, H_C$).\n*   $G, H, J$ are the intersection points of the nine-point circle with $\\overline{BD}$, $\\overline{AE}$, $\\overline{AF}$ re

TOT gets the answer correct

In [15]:
problem_rows[TOT][TOTAL_TOKENS] = 213862

problem_rows[TOT][CORRECT] = True
problem_rows[TOT][PARAMS] = "all=3"


### Our method: Basic form [Yes]

In [22]:
from multi_agent.multi_agent import Role, Problem, conversation_google, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
)

messages, raw, path = conversation_google(model=model, name="gemini_25_flash_think_Problem1_attmp1" ,n_steps=10, problem=problem)
rank_google_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 


Agent: Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize an

Our methd also got the right answer

In [16]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 200213
problem_rows[RS_BASIC][CORRECT] = True
problem_rows[RS_BASIC][PARAMS] = "n_steps=10"

## Our method: Summarized [Yes]

In [ ]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_google
from multi_agent.multi_agent import Role, Problem, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient
from importlib import reload
reload(summerized_conv)

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=first_problem_gold_answer
  
)

msg = summarized_rejection_sampling_google(
  model=model,
  name=f"{model_name}_problem_1",
  n_steps=5,
  problem=problem
)

rank_google_answer(model=model, conversation=msg)

Iteration 0

Solver: 
The problem asks us to find the value of $\wideparen{DE} + 2 \cdot \wideparen{HJ} + 3 \cdot \wideparen{FG}$ for a triangle $\triangle ABC$ with angles $\angle A = 84^\circ$, $\angle B = 60^\circ$, and $\angle C = 36^\circ$. Points $D, E, F$ are the midpoints of sides $\overline{BC}$, $\overline{AC}$, and $\overline{AB}$, respectively. The circumcircle of $\triangle DEF$ (which is the nine-point circle of $\triangle ABC$) intersects $\overline{BD}$, $\overline{AE}$, and $\overline{AF}$ at points $G$, $H$, and $J$, respectively.

First, let's identify the points $G, H, J$.
The circumcircle of $\triangle DEF$ is the nine-point circle of $\triangle ABC$. This circle passes through the midpoints of the sides ($D, E, F$) and the feet of the altitudes ($H_A, H_B, H_C$).
1.  **Point G**: $G$ is the intersection of the circumcircle of $\triangle DEF$ and the segment $\overline{BD}$. $D$ is already on the circumcircle. The line containing $\overline{BD}$ is the side $\overl

('Welcome Rejecter,\n\nI have thoroughly reviewed all suggested answers and the extensive reasoning that led to them, with a specific focus on the core rejections and your guidance regarding a "fundamental misunderstanding."\n\nHere\'s my analysis, argument for plausibility for each answer, and final ranking:\n\n---\n\n### Analysis of the Problem\'s Conditions and My Derivations:\n\nMy core derivation for 336 relies on the following chain of reasoning:\n\n1.  **Nine-Point Circle Identification:** The circumcircle of $\\triangle DEF$ (where $D, E, F$ are midpoints of sides) is unequivocally the nine-point circle of $\\triangle ABC$. This is a standard geometric fact.\n2.  **Identification of G, H, J:**\n    *   The nine-point circle passes through the midpoints of sides ($D, E, F$) and the feet of the altitudes ($H_A, H_B, H_C$).\n    *   $G$ is a point where the nine-point circle intersects $\\overline{BD}$. Since $D$ is the midpoint of $BC$, $\\overline{BD}$ is a segment of $\\overlin

Summarized got the right answer

In [17]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  367313
problem_rows[RS_SUM][CORRECT] =  True
problem_rows[RS_SUM][PARAMS] =  "n_steps=5"


## Token cost

In [18]:

for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/cmimc_2025: 16,5,reflexion,49787,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 16,5,tot,1612774,False,all=4,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_basic,597448,False,n_steps=20,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_summarized,538780,False,n_steps=10,2nd
0,MathArena/aime_2025: 20,4,reflexion,66965,True,max_attempt=3,NaN
0,MathArena/aime_2025: 20,4,tot,213862,True,all=3,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_basic,200213,True,n_steps=10,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_summarized,367313,True,n_steps=5,NaN


# Problem 3, difficulty level 3

In [19]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==3].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned

unique_problem_label                          MathArena/hmmt_feb_2025: 18
answer                  Let $N=2025$ and $T=100$. The numbers are $a_1...
gold_answer                                              \frac{2025}{101}
ten_percentile_group                                                    3
problem                 Albert writes $2025$ numbers $a_{1}, \ldots, a...
Name: 13604, dtype: object

In [20]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 18',
  'ten_percentile_group': 3},
 'tot': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 18',
  'ten_percentile_group': 3},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 18',
  'ten_percentile_group': 3},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/hmmt_feb_2025: 18',
  'ten_percentile_group': 3}}

In [21]:
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)



Albert writes $2025$ numbers $a_{1}, \ldots, a_{2025}$ in a circle on a
blackboard. Initially, each of the numbers is uniformly and independently
sampled at random from the interval $[0,1]$. Then, each second, he
\emph{simultaneously} replaces $a_{i}$ with $\max \left(a_{i-1}, a_{i},
a_{i+1}\right)$ for all $i=1,2, \ldots, 2025$ (where $a_{0}=a_{2025}$ and
$a_{2026}=a_{1}$ ). Compute the expected value of the number of distinct values
remaining after $100$ seconds.
answer \frac{2025}{101}


### Reflextion: [Yes]


In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

from models.google_api import GoogleClient, google_user_format
model_name="gemini-2.5-flash"


client = GoogleClient()

model = client.select_model(
  model_name=model_name

)

eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3,
    user_format=google_user_format
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
Let $N=2025$ be the total number of initial values and $T=100$ be the number of seconds.
Let $a_1^{(0)}, \ldots, a_N^{(0)}$ be the initial values, sampled independently and uniformly from $[0,1]$.
At each second $t$, the values are updated according to $a_i^{(t)} = \max(a_{i-1}^{(t-1)}, a_i^{(t-1)}, a_{i+1}^{(t-1)})$, where indices are taken modulo $N$.

First, let's determine the value of $a_i^{(T)}$ in terms of the initial values.
For $t=1$, $a_i^{(1)} = \max(a_{i-1}^{(0)}, a_i^{(0)}, a_{i+1}^{(0)})$.
For $t=2$, $a_i^{(2)} = \max(a_{i-1}^{(1)}, a_i^{(1)}, a_{i+1}^{(1)})$. Substituting the expressions for $a_j^{(1)}$:
$a_i^{(2)} = \max(\max(a_{i-2}^{(0)}, a_{i-1}^{(0)}, a_i^{(0)}), \max(a_{i-1}^{(0)}, a_i^{(0)}, a_{i+1}^{(0)}), \max(a_i^{(0)}, a_{i+1}^{(0)}, a_{i+2}^{(0)}))$.
This simplifies to $a_i^{(2)} = \max(a_{i-2}^{(0)}, a_{i-1}^{(0)}, a_i^{(0)}, a_{i+1}^{(0)}, a_{i+2}^{(0)})$.
By induction, after $T$ seconds, the value $a_i^{(T)}$ is the maximum of the $2T+1$

Reflextion gets the correct answer


In [22]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 24691 + 6441

problem_rows[REFLEXION][CORRECT] = True
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"


### TOT [No]

In [8]:


from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from google import genai
model_name="gemini-2.5-flash"
config = ToTConfig(
    key_env_name="GEMINI_API_KEY",
    endpoint_env_name= None,
    model_name=model_name,
    n_evaluate_sample=3,
    n_select_sample=3,
    n_generate_sample=3,
    steps=3,
    api_version=None,
    client_type=genai.Client
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import google_usage
usage = google_usage()
print(usage)

>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
vote no match: ['The problem asks for the expected number of distinct values after $t=100$ seconds, where $N=2025$ elements are arranged in a circle. The update rule is $a_i^{(s+1)} = \\max \\left(a_{i-1}^{(s)}, a_{i}^{(s)}, a_{i+1}^{(s)}\\right)$. The initial values $X_j = a_j^{(0)}$ are i.i.d. continuous random variables.\n\nFirst, we determine the form of $a_i^{(t)}$ after $t$ seconds.\nAfter 1 second: $a_i^{(1)} = \\max(X_{i-1}, X_i, X_{i+1})$.\nAfter 2 seconds: $a_i^{(2)} = \\max(a_{i-1}^{(1)}, a_i^{(1)}, a_{i+1}^{(1)}) = \\max(\\max(X_{i-2}, X_{i-1}, X_i), \\max(X_{i-1}, X_i, X_{i+1}), \\max(X_i, X_{i+1}, X_{i+2})) = \\max(X_{i-2}, X_{i-1}, X_i, X_{i+1}, X_{i+2})$.\nBy induction, after $t$ seconds, $a_i^{(t)} = \\max(X_{i-t}, \\ldots, X_{i+t})$.\nThe values $Y_i = a_i^{(t)}$ are the maximums of sliding windows of size $w = 2t+1$.\nIn th

Tot does not get the answer correct 

In [23]:
problem_rows[TOT][TOTAL_TOKENS] = 456549

problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=3"

### Our method: Basic form [Yes]

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation_google, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
)

messages, raw, path = conversation_google(model=model, name="gemini_25_flash_think_Problem1_attmp1" ,n_steps=10, problem=problem)
rank_google_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 


Agent: Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize an

Our methods raks the crrect answer first

In [24]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 43833 + 6499 + 36084 + 14213 + 26744 + 9917 + 47733 + 13369 + 37802 + 15234 + 26308
problem_rows[RS_BASIC][CORRECT] = True
problem_rows[RS_BASIC][PARAMS] = "n_steps=10"

### our method: Summarized [2nd]

In [ ]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_google
from multi_agent.multi_agent import Role, Problem, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient
from importlib import reload
reload(summerized_conv)

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=first_problem_gold_answer
  
)

msg = summarized_rejection_sampling_google(
  model=model,
  name=f"{model_name}_problem_1",
  n_steps=5,
  problem=problem
)

rank_google_answer(model=model, conversation=msg)

Iteration 0

Solver: 
Hello, I'm Solver. I'm ready to tackle this problem.

The problem asks for the expected value of the number of distinct values remaining after 100 seconds. We have $N=2025$ numbers, $a_1, \ldots, a_N$, arranged in a circle. Initially, each $a_i$ is drawn independently from $U[0,1]$. Each second, all $a_i$ are simultaneously replaced by $\max(a_{i-1}, a_i, a_{i+1})$, using circular indexing.

Let $a_i^{(t)}$ denote the value of $a_i$ at time $t$. The update rule is $a_i^{(t+1)} = \max(a_{i-1}^{(t)}, a_i^{(t)}, a_{i+1}^{(t)})$.

First, let's determine the value of $a_i^{(T)}$ after $T=100$ seconds.
For $t=1$: $a_i^{(1)} = \max(a_{i-1}^{(0)}, a_i^{(0)}, a_{i+1}^{(0)})$.
For $t=2$: $a_i^{(2)} = \max(a_{i-1}^{(1)}, a_i^{(1)}, a_{i+1}^{(1)})$.
Substituting the expressions for $a_j^{(1)}$:
$a_i^{(2)} = \max(\max(a_{i-2}^{(0)}, a_{i-1}^{(0)}, a_i^{(0)}), \max(a_{i-1}^{(0)}, a_i^{(0)}, a_{i+1}^{(0)}), \max(a_i^{(0)}, a_{i+1}^{(0)}, a_{i+2}^{(0)}))$.
Since the maximum opera

('Hello Solver and Rejecter!\n\nThank you for this opportunity to revisit the previous answers and reflect on the knowledge acquired. The process of deep diving into the underlying theory has been invaluable.\n\nHere\'s an analysis of each submitted answer, arguing for its potential validity, and then a final ranking based on plausibility.\n\n---\n\n### Problem Recap:\n\n*   **$N = 2025$** numbers $a_1, \\ldots, a_N$ in a circle, initially i.i.d. $U[0,1]$.\n*   Each second, $a_i$ is replaced by $\\max(a_{i-1}, a_i, a_{i+1})$.\n*   After **$T=100$** seconds, $a_i^{(T)} = \\max_{j=i-T}^{i+T} a_j^{(0)}$.\n*   The effective window size is $K = 2T+1 = 201$.\n*   We need the expected number of *distinct values* in the final array $\\{a_1^{(T)}, \\ldots, a_N^{(T)}\\}$. An initial value $X_j$ is distinct if it is the maximum in at least one of the $K$-sized windows it influences.\n\n---\n\n### Analysis of Previous Answers:\n\n**1. Answer: 675/67**\n\n*   **Derivation:** $N/(2T+1) = 2025/(2 \\t

In [25]:
problem_rows[RS_SUM][TOTAL_TOKENS] = 225746
problem_rows[RS_SUM][CORRECT] =  True
problem_rows[RS_SUM][PARAMS] =  "n_steps=5"


In [26]:
## Problem 
for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/cmimc_2025: 16,5,reflexion,49787,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 16,5,tot,1612774,False,all=4,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_basic,597448,False,n_steps=20,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_summarized,538780,False,n_steps=10,2nd
0,MathArena/aime_2025: 20,4,reflexion,66965,True,max_attempt=3,NaN
0,MathArena/aime_2025: 20,4,tot,213862,True,all=3,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_basic,200213,True,n_steps=10,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_summarized,367313,True,n_steps=5,NaN
0,MathArena/hmmt_feb_2025: 18,3,reflexion,31132,True,max_attempt=3,NaN
0,MathArena/hmmt_feb_2025: 18,3,tot,456549,False,all=3,NaN


# Problem 4, diff 2

In [27]:
import pandas as pd
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==2].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned
import textwrap

problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)



Alex divides a disk into four quadrants with two perpendicular diameters
intersecting at the center of the disk. He draws $25$ more lines segments
through the disk, drawing each segment by selecting two points at random on the
perimeter of the disk in different quadrants and connecting those two points.
Find the expected number of regions into which these $27$ line segments divide
the disk.
answer 204


In [28]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/aime_2025: 13',
  'ten_percentile_group': 2},
 'tot': {'unique_problem_label': 'MathArena/aime_2025: 13',
  'ten_percentile_group': 2},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/aime_2025: 13',
  'ten_percentile_group': 2},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/aime_2025: 13',
  'ten_percentile_group': 2}}

### Reflextion[No]


In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

from models.google_api import GoogleClient, google_user_format
model_name="gemini-2.5-flash"


client = GoogleClient()

model = client.select_model(
  model_name=model_name

)

eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3,
    user_format=google_user_format
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print("Number of tokens used", model.num_tokens)


=== Attempt 1 ===
The problem asks for the expected number of regions into which 27 line segments divide a disk.
Let $R$ be the number of regions. The formula for the number of regions created by $L$ line segments inside a disk, assuming endpoints are on the perimeter and segments are in general position (no three segments intersect at the same point, segments don't pass through existing intersection points unless specified), is $R = 1 + L + I$, where $L$ is the number of segments and $I$ is the number of internal intersection points.

In this problem, $L = 27$ (2 perpendicular diameters + 25 additional segments).
We need to find the expected number of internal intersection points, $E[I]$.
By linearity of expectation, $E[R] = 1 + L + E[I]$.

Let the two perpendicular diameters be $d_1$ and $d_2$. Let the 25 additional segments be $s_1, \dots, s_{25}$.
The intersection points can be of three types:
1.  Intersection between $d_1$ and $d_2$. There is exactly one such point (the center of

Not right for reflextion

In [29]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 34768 + 17741 + 10363 + 15011 + 8807

problem_rows[REFLEXION][CORRECT] = False
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"


### Tot [No]

In [5]:


from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from google import genai
model_name="gemini-2.5-flash"
config = ToTConfig(
    key_env_name="GEMINI_API_KEY",
    endpoint_env_name= None,
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=None,
    client_type=genai.Client
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import google_usage
usage = google_usage()
print(usage)

path c:\Users\Thomas\Documents\Datalogi\Kandidat\9. semester\9. semester notes\NLP\Project\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
vote no match: ['The problem asks for the expected number of regions formed by 27 line segments in a disk.\nThe formula for the number of regions $R$ created by $L$ line segments in a convex region (like a disk), where the segments are in general position (no three concurrent), is $R = L + I + 1$, where $I$ is the number of internal intersection points.\n\nIn this problem, the total number of line segments is $L = 2$ (initial diameters) + $25$ (new segments) = $27$.\nSo, the number of regions is $R = 27 + I + 1 = 28 + I$.\nWe are looking for the expected number of regions, $E[R]$. By the linearity of expectation, $E[R] = E[28 + I] = 28 + E[I]$.\n\nTherefore, the main task is to calculate $E[I]$, the expected total number of intersection poi

Tot does not get the quesiton correct

In [30]:
problem_rows[TOT][TOTAL_TOKENS] = 979913

problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=4"

### our method: Basic form [No]

In [ ]:
from multi_agent.multi_agent import Role, Problem, conversation_google, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
)

messages, raw, path = conversation_google(model=model, name="gemini_25_flash_think_Problem1_attmp1" ,n_steps=20, problem=problem)
rank_google_answer(model=model, conversation=messages)
print("Number of tokens used", model.num_tokens)


STEP 0: 


Agent: Role: Solver
 You solve problems.  You try to reason step by step. You are not too confident
in your answers (in the sense you are open to be wrong), but rather you rely on
fully fleshed out mathematical reasoning.  You try to explore many ideas.
Everytime you speak you will propose a fresh answer.  You dont submit the same
answer twice. Everytime you come with a new answer, you state all the previous
answers in a list in format of tuples: (Answer, short summary).  For example,  [
(780, induction on N, and lower bound on Z/N), (28/2, CLT of H and proof by
contradiction of Z>N) ] Then you check that your new proposal is not in that
list. If it is, you try again.  Use the early parts of your prompt as thinking
text, not "for science paper style" - meaning you can write your things and
doubts. Ex "I am thinking there could be a hint in the upper bound. I will check
it out. Ahh, I see I made a mistake. But now the size formula seems really
promising!"  "Then formalize an

In [31]:
problem_rows[RS_BASIC][TOTAL_TOKENS] = 25996 + 3665 + 16125 + 6045 + 29176 + 12097 + 23930 + 13140 + 23370 + 14342 + 26449 + 15888 + 28331 + 17102 + 28000 + 20038 + 27415 + 21215 + 26131 + 21829 + 26769
problem_rows[RS_BASIC][CORRECT] = False
problem_rows[RS_BASIC][PARAMS] = "n_steps=20"
problem_rows[RS_BASIC][RESULT_DESC] = None


### Our metod: Summarized [11th]

In [ ]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_google
from multi_agent.multi_agent import Role, Problem, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient
from importlib import reload
reload(summerized_conv)

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=first_problem_gold_answer
  
)

msg = summarized_rejection_sampling_google(
  model=model,
  name=f"{model_name}_problem_5",
  n_steps=20,
  problem=problem
)

rank_google_answer(model=model, conversation=msg)

Iteration 0

Solver: 
Current conversation summary:
The problem asks for the expected number of regions created by 27 line segments in a disk: two initial perpendicular diameters and 25 additional segments. Each additional segment connects two points chosen randomly from the perimeter, with each point in a different quadrant.

Current suggested answers:
[]

Let $N_{total}$ be the total number of line segments, which is $2 + 25 = 27$.
Let $D_1$ and $D_2$ be the two initial perpendicular diameters. They intersect at the center of the disk. These two diameters divide the disk into 4 regions.
Let $L_1, \dots, L_{25}$ be the 25 additional line segments. Each $L_k$ is drawn by selecting two points at random on the perimeter in different quadrants and connecting them.

We use the general formula for the number of regions $R$ created by segments in a disk. If we start with an empty disk (1 region) and add segments one by one, $R_j = R_{j-1} + 1 + I_j$, where $I_j$ is the number of new internal

('The challenge is to identify the single correct answer among a list of previously rejected ones, using the cumulative knowledge acquired during the conversation, especially the "firmly accepted components" and the Rejecter\'s specific feedback.\n\nThe formula for the expected number of regions, $E[R]$, is:\n$E[R] = R_0 + N(1 + E[\\text{intersections with diameters}]) + \\binom{N}{2} P(L_k \\cap L_j)$\n\nFrom the "firmly accepted components":\n*   $R_0 = 4$\n*   $N = 25$\n*   $E[\\text{intersections with diameters}] = 4/3$\n\nSubstituting these, the first two terms are:\n$4 + 25(1 + 4/3) = 4 + 25(7/3) = 4 + 175/3 = (12 + 175)/3 = 187/3$.\n\nThe equation becomes: $E[R] = 187/3 + \\binom{25}{2} P(L_k \\cap L_j)$\n$\\binom{25}{2} = \\frac{25 \\times 24}{2} = 300$.\nSo, $E[R] = 187/3 + 300 \\times P(L_k \\cap L_j)$.\n\nThe crucial part is determining $P(L_k \\cap L_j)$, the probability that any two distinct chords $L_k$ and $L_j$ intersect. The four endpoints $A_k, B_k, A_j, B_j$ can each

Rankes true anser 11th

In [32]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  1130850
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][PARAMS] =  "n_steps=20"
problem_rows[RS_SUM][RESULT_DESC] =  "11th"

## Tokens

In [33]:

for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats


,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/cmimc_2025: 16,5,reflexion,49787,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 16,5,tot,1612774,False,all=4,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_basic,597448,False,n_steps=20,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_summarized,538780,False,n_steps=10,2nd
0,MathArena/aime_2025: 20,4,reflexion,66965,True,max_attempt=3,NaN
0,MathArena/aime_2025: 20,4,tot,213862,True,all=3,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_basic,200213,True,n_steps=10,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_summarized,367313,True,n_steps=5,NaN
0,MathArena/hmmt_feb_2025: 18,3,reflexion,31132,True,max_attempt=3,NaN
0,MathArena/hmmt_feb_2025: 18,3,tot,456549,False,all=3,NaN


# problem 5 - diff 1

In [34]:
import pandas as pd
import textwrap
DIFFICULTY = "ten_percentile_group"
df_pruned = df[df[DIFFICULTY]==1].iloc[0][["unique_problem_label", "answer", "gold_answer", "ten_percentile_group", "problem"]]
df_pruned
problem_description = df_pruned["problem"]

problem_gold_answer = df_pruned["gold_answer"]
print(textwrap.fill(text=problem_description, width=80))
print("answer", problem_gold_answer)


Let $ABCDE$ be a convex pentagon with $AB=14$, $BC=7$, $CD=24$, $DE=13$,
$EA=26$, and $\angle B=\angle E=60^{\circ}$. For each point $X$ in the plane,
define $f(X)=AX+BX+CX+DX+EX$. The least possible value of $f(X)$ can be
expressed as $m+n\sqrt{p}$, where $m$ and $n$ are positive integers and $p$ is
not divisible by the square of any prime. Find $m+n+p$.
answer 60


In [35]:
stat_dict = {
  UNIQUE_PROBLEM_LABEL: df_pruned[UNIQUE_PROBLEM_LABEL],
  DIFFICULTY: df_pruned[DIFFICULTY]
  }
stat_dict

problem_rows = {
  REFLEXION: stat_dict.copy(),
  TOT: stat_dict.copy(),
  RS_BASIC: stat_dict.copy(),
  RS_SUM: stat_dict.copy()
  }

problem_rows

{'reflexion': {'unique_problem_label': 'MathArena/aime_2025: 14',
  'ten_percentile_group': 1},
 'tot': {'unique_problem_label': 'MathArena/aime_2025: 14',
  'ten_percentile_group': 1},
 'rejection_sampling_basic': {'unique_problem_label': 'MathArena/aime_2025: 14',
  'ten_percentile_group': 1},
 'rejection_sampling_summarized': {'unique_problem_label': 'MathArena/aime_2025: 14',
  'ten_percentile_group': 1}}

## Reflextion [No]

In [ ]:
from multi_agent.multi_agent import Problem
from models.prompt_template import Reflexion_Solver, Reflector
from baselines.reflexion.reflexion import ReflexionAgent, ReflexionStrategy, evaluator_fn
from functools import partial
problem = problem = Problem(
    roles=[Reflexion_Solver, Reflector],  # add Solver if you have one
    problem_descr=problem_description,
    answer=problem_gold_answer
)

from models.google_api import GoogleClient, google_user_format
model_name="gemini-2.5-flash"


client = GoogleClient()

model = client.select_model(
  model_name=model_name

)

eval = partial(evaluator_fn, model=model)

agent = ReflexionAgent(
    llm=model,
    strategy=ReflexionStrategy.LAST_ATTEMPT_AND_REFLEXION,
    evaluator_fn=eval,
    reflector_prompt=Reflector,
    max_attempts=3,
    user_format=google_user_format
)

solution = agent.solve(problem.problem_description)
print("\nFINAL SOLUTION:\n", solution)
print(model.compute_token_cost())


=== Attempt 1 ===
Let the given side lengths be $AB=a=14$, $BC=b=7$, $CD=c=24$, $DE=d=13$, $EA=e=26$.
The given angles are $\angle B = \angle ABC = 60^{\circ}$ and $\angle E = \angle DEA = 60^{\circ}$.
We want to find the minimum value of $f(X)=AX+BX+CX+DX+EX$.

This is a problem that can be solved using rotations, similar to the Fermat point problem for a triangle.
Consider the terms $AX+BX+CX$. The sum $AX+BX+CX$ is minimized when $X$ is the Fermat point of $\triangle ABC$.
Since $\angle B = 60^{\circ}$, we can calculate the length of $AC$ using the Law of Cosines in $\triangle ABC$:
$AC^2 = AB^2 + BC^2 - 2(AB)(BC)\cos(\angle B)$
$AC^2 = 14^2 + 7^2 - 2(14)(7)\cos(60^{\circ})$
$AC^2 = 196 + 49 - 2(14)(7)(1/2)$
$AC^2 = 196 + 49 - 98 = 147$.
So $AC = \sqrt{147} = \sqrt{49 \times 3} = 7\sqrt{3}$.

Similarly, consider the terms $DX+EX+AX$. The sum $DX+EX+AX$ is minimized when $X$ is the Fermat point of $\triangle DEA$.
We can calculate the length of $AD$ using the Law of Cosines in $\tri

reflexion does not get the right answer

In [36]:
problem_rows[REFLEXION][TOTAL_TOKENS] = 52122

problem_rows[REFLEXION][CORRECT] = False
problem_rows[REFLEXION][PARAMS] = "max_attempt=3"


## TOT [No]


In [5]:


from baselines.tot.math_arena_tot_setup import ToTConfig, run_math_arena_tot
from google import genai
model_name="gemini-2.5-flash"
config = ToTConfig(
    key_env_name="GEMINI_API_KEY",
    endpoint_env_name= None,
    model_name=model_name,
    n_evaluate_sample=4,
    n_select_sample=4,
    n_generate_sample=4,
    steps=4,
    api_version=None,
    client_type=genai.Client
  )

ys, infos = run_math_arena_tot(ToTConfig=config, problem_descr=problem_description, answer=problem_gold_answer)

print(ys)
from baselines.tot.tree_of_thought_llm_master.src.tot.models import google_usage
usage = google_usage()
print(usage)

path c:\Users\Thomas\Documents\Datalogi\Kandidat\9. semester\9. semester notes\NLP\Project\NLP-group-15
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
>>>tries to call client.. Attempt> 0 <<<
>>>succesfully called client<<<
vote no match: ['The problem asks for the minimum value of $f(X)=AX+BX+CX+DX+EX$ for a convex pentagon $ABCDE$ with given side lengths $AB=14$, $BC=7$, $CD=24$, $DE=13$, $EA=26$, and angles $\\angle B=60^{\\circ}$, $\\angle E=60^{\\circ}$. The minimum value is of the form $m+n\\sqrt{p}$, where $m,n$ are positive integers and $p$ is not divisible by the square of any prime. We need to calculate $m+n+p$.\n\nLet\'s analyze the properties of the triangles involving the $60^{\\circ}$ angles:\n\n1.  **Triangle ABC**: Given $AB=14$, $BC=7$, $\\angle B=60^{\\circ}$.\n    Using the Law of Cosines to find $AC$:\n    $AC^2 = AB^2 + BC^2 - 2(AB)(BC)\\cos 60^{\\circ}$\n    $AC^2 = 14^2 + 7^2 - 2(14)(7)(1/2)$\n    $AC^2 = 196 + 49 - 98 = 147$\n    $AC = 

tot does not get it right

In [37]:
problem_rows[TOT][TOTAL_TOKENS] = 1195735

problem_rows[TOT][CORRECT] = False
problem_rows[TOT][PARAMS] = "all=4"

## our method: summarized [Yes, but not even in ranking]

In [ ]:
import summerized_conv
from summerized_conv import summarized_rejection_sampling_google
from multi_agent.multi_agent import Role, Problem, rank_google_answer
from models.prompt_template import Solver, Rejector
from models.google_api import GoogleClient
from importlib import reload
reload(summerized_conv)

client = GoogleClient()
model_name="gemini-2.5-flash"
model = client.select_model(
  model_name=model_name

)

problem = Problem(
  roles=[Solver, Rejector],
  problem_descr=problem_description, 
  answer=problem_gold_answer
  
)

msg = summarized_rejection_sampling_google(
  model=model,
  name=f"{model_name}_problem_5",
  n_steps=30,
  problem=problem
)

rank_google_answer(model=model, conversation=msg)

Iteration 0

Solver: 
The conversation so far has focused on understanding the problem statement and exploring various geometric approaches, primarily revolving around the concept of the Fermat point and "unfolding" techniques for sums of distances.
We've identified the key features: a convex pentagon with specific side lengths and two $60^\circ$ internal angles ($\angle B$ and $\angle E$).
A significant discovery was that the triangles $\triangle ABC$ and $\triangle DEA$ are $30-60-90$ triangles, implying $\angle BCA = 90^\circ$ and $\angle ADE = 90^\circ$.
We've attempted various coordinate system setups, but they led to inconsistencies with the given side length $CD=24$.
We also explored the properties of the sum of distances $f(X) = AX+BX+CX+DX+EX$ and potential transformations using rotations around vertices with $60^\circ$ angles. This led to values like $AC_1=7$ and $AD_1=13\sqrt{3}$ from these rotations.
We have not yet proposed a final answer.

Let's carefully analyze the prob

('Here\'s an updated summary of our conversation so far.\n\n**Latest Interaction Summary:**\nThe Solver successfully provided a new and unique numerical sum ($m+n+p=31$). The calculation of the irrational component ($n=20, p=3$) and the key geometric properties (lengths $AC=7\\sqrt{3}$, $AD=13\\sqrt{3}$, and angles $\\angle CDE=120^{\\circ}$, $\\angle BCD \\approx 154.5^{\\circ}$) were validated as accurate. However, the Rejecter once again found the derivation for the integer component \'m\' (leading to $m=8$) to be insufficient. The Solver used a formula ($m = CD - AC_1 - AD_1 + \\delta$) which was *stated* and applied, rather than being *rigorously derived* from a full, explicit geometric unfolding of the *entire sum* $f(X)=AX+BX+CX+DX+EX$. The Rejecter emphasized that the connection between the rotations and how *all five segments* transform into a single, continuous, straight line path (or a sequence of such paths) that precisely equals the stated formula for \'m\' was not clearly

In [ ]:
problem_rows[RS_SUM][TOTAL_TOKENS] =  1665394
problem_rows[RS_SUM][CORRECT] =  False
problem_rows[RS_SUM][RESULT_DESC] =  "Non in top5, model does not rank"
problem_rows[RS_SUM][PARAMS] =  "n_steps=30"


for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

In [39]:
for method, row in problem_rows.items():
  df_row = pd.DataFrame([row])
  df_row[METHOD] = method
  df_stats = pd.concat([df_stats, df_row])
df_stats

,unique_problem_label,ten_percentile_group,method,total_tokens,correct,params,result_description
0,MathArena/cmimc_2025: 16,5,reflexion,49787,False,max_attempt=3,NaN
0,MathArena/cmimc_2025: 16,5,tot,1612774,False,all=4,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_basic,597448,False,n_steps=20,NaN
0,MathArena/cmimc_2025: 16,5,rejection_sampling_summarized,538780,False,n_steps=10,2nd
0,MathArena/aime_2025: 20,4,reflexion,66965,True,max_attempt=3,NaN
0,MathArena/aime_2025: 20,4,tot,213862,True,all=3,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_basic,200213,True,n_steps=10,NaN
0,MathArena/aime_2025: 20,4,rejection_sampling_summarized,367313,True,n_steps=5,NaN
0,MathArena/hmmt_feb_2025: 18,3,reflexion,31132,True,max_attempt=3,NaN
0,MathArena/hmmt_feb_2025: 18,3,tot,456549,False,all=3,NaN


In [40]:
df_stats.to_csv("results/model_stats/gemini_25_flash.csv")
